# Imports

In [ ]:
import json
import re
from typing import Any, Dict, Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Plotting Settings

In [ ]:
###########################
# Setup Plotting Defaults #
###########################
# For more options see https://matplotlib.org/users/customizing.html

# Commands for high detail plots (much larger in file size though)
# mpl.rcParams['agg.path.chunksize'] = 1000
# mpl.rcParams['savefig.dpi'] = 1000

# Increase display resolution
mpl.rcParams["figure.dpi"] = 100

# Line styles
mpl.rcParams["lines.linewidth"] = 1.5
# prevents lines from being jagged
mpl.rcParams["lines.antialiased"] = True

#
mpl.rcParams["lines.dashed_pattern"] = 2.8, 1.5
mpl.rcParams["lines.dashdot_pattern"] = 4.8, 1.5, 0.8, 1.5

# I have no idea what this does
mpl.rcParams["lines.dotted_pattern"] = 1.1, 1.1

mpl.rcParams["lines.scale_dashes"] = True

# Default colors
from cycler import cycler

# defines the colors to cycle through when line colors are not specifically defined
mpl.rcParams["axes.prop_cycle"] = cycler(
    "color", ["cornflowerblue", "forestgreen", "maroon", "goldenrod", "firebrick", "mediumorchid"]
)


# Fonts
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = "DejaVu Serif"
mpl.rcParams["font.sans-serif"] = "DejaVu Sans"
# mpl.rcParams['text.usetex'] = True

# Axes
mpl.rcParams["axes.linewidth"] = 1.0
mpl.rcParams["axes.labelsize"] = 25
mpl.rcParams["axes.labelpad"] = 9.0

# Title
mpl.rcParams["axes.titlepad"] = 10.0
mpl.rcParams["axes.titlesize"] = 25


# Tick marks - the essence of life
mpl.rcParams["xtick.top"] = True
mpl.rcParams["xtick.major.size"] = 5
mpl.rcParams["xtick.minor.size"] = 2.5
mpl.rcParams["xtick.major.width"] = 1.0
mpl.rcParams["xtick.minor.width"] = 0.75
mpl.rcParams["xtick.major.pad"] = 8
mpl.rcParams["xtick.labelsize"] = 22

# default in mpl v2.0 is 'out'
mpl.rcParams["xtick.direction"] = "in"
mpl.rcParams["ytick.direction"] = "in"

mpl.rcParams["xtick.minor.visible"] = True
mpl.rcParams["ytick.right"] = True
mpl.rcParams["ytick.major.size"] = 5
mpl.rcParams["ytick.minor.size"] = 2.5
mpl.rcParams["ytick.major.width"] = 1.0
mpl.rcParams["ytick.minor.width"] = 0.75
mpl.rcParams["ytick.major.pad"] = 8
mpl.rcParams["ytick.labelsize"] = 22
mpl.rcParams["ytick.minor.visible"] = True

# Error bar plots
# default in mpl v2.0 is no caps on error bars
mpl.rcParams["errorbar.capsize"] = 3

# Legend
mpl.rcParams["legend.fontsize"] = 22
mpl.rcParams["legend.frameon"] = True
mpl.rcParams["legend.framealpha"] = 0.8
mpl.rcParams["legend.edgecolor"] = "black"
mpl.rcParams["legend.fancybox"] = True
mpl.rcParams["legend.borderpad"] = 0.4  # border whitespace
mpl.rcParams["legend.labelspacing"] = 0.5  # the vertical space between the legend entries
mpl.rcParams["legend.handlelength"] = 1.5  # the length of the legend lines
mpl.rcParams["legend.handleheight"] = 0.7  # the height of the legend handle
mpl.rcParams["legend.handletextpad"] = 0.5  # the space between the legend line and legend text
mpl.rcParams["legend.borderaxespad"] = 0.5  # the border between the axes and legend edge
mpl.rcParams["legend.columnspacing"] = 2.0  # column separation

# Figure size
mpl.rcParams["figure.figsize"] = 8, 8

# Save details
mpl.rcParams["savefig.bbox"] = "tight"
mpl.rcParams["savefig.pad_inches"] = 0.1
mpl.rcParams["savefig.dpi"] = 200  # higher-res than default 100 dpi

# tex packages
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amssymb}"

# Helpers 

In [ ]:
def extract_micro_metric_arrays(
    results: Dict[str, Any],
    model_size: str = "micro",
    metrics: Iterable[str] = ("auc", "avg_loss", "inv_bkg_eff"),
) -> Dict[str, Dict[str, Any]]:
    """
    Given a dict loaded from Top-Class-Finetuned-Model-Paths-Eval.json,
    extract per-pretraining-mode metric tensors for the micro model.

    Returns
    -------
    out : dict
        out[pretrain_mode] = {
            "mean": np.ndarray,
            "std": np.ndarray,
            "top_sizes": List[str],
            "pretrain_sizes": List[int] | None
        }

        Shapes:
          - For modes != "from_scratch":
              mean, std: (n_top_sizes, n_pretrain_sizes, n_metrics)
          - For mode == "from_scratch":
              mean, std: (n_top_sizes, n_metrics)
    """
    metrics = tuple(metrics)

    # Parse "1k-Top", "10k-Top", "100k-Top", "1M-Top" -> numeric to sort
    def _parse_top_key(k: str) -> int:
        m = re.match(r"(\d+)([kM])?-Top", k)
        if not m:
            return 0
        val = int(m.group(1))
        suffix = m.group(2)
        if suffix == "k":
            val *= 1_000
        elif suffix == "M":
            val *= 1_000_000
        return val

    # Only keep top-keys that actually have `model_size`
    top_keys = [k for k, v in results.items() if isinstance(v, dict) and model_size in v]
    top_keys = sorted(top_keys, key=_parse_top_key)
    n_top = len(top_keys)

    # Collect all pretraining modes and all pretraining sizes (exclude from_scratch from sizes)
    pretrain_modes = set()
    pretrain_sizes_set = set()

    for top in top_keys:
        micro_block = results[top][model_size]
        for mode, mode_block in micro_block.items():
            pretrain_modes.add(mode)
            if mode == "from_scratch":
                continue
            if isinstance(mode_block, dict):
                for size_key in mode_block.keys():
                    try:
                        pretrain_sizes_set.add(int(size_key))
                    except ValueError:
                        pass

    pretrain_modes = sorted(
        pretrain_modes
    )  # e.g. ['classifier', 'classifier+generator', 'from_scratch', 'generator']

    if -1 in pretrain_sizes_set:
        positive_sizes = sorted(s for s in pretrain_sizes_set if s != -1)
        pretrain_sizes = positive_sizes + [-1]
    else:
        pretrain_sizes = sorted(pretrain_sizes_set)

    n_pre = len(pretrain_sizes)
    n_metrics = len(metrics)

    out: Dict[str, Dict[str, Any]] = {}

    for mode in pretrain_modes:
        if mode == "from_scratch":
            # Special: no pretraining-size axis → 2D (n_top, n_metrics)
            mean_arr = np.full((n_top, n_metrics), np.nan, dtype=float)
            std_arr = np.full((n_top, n_metrics), np.nan, dtype=float)

            for i, top in enumerate(top_keys):
                micro_block = results[top][model_size]
                fs_block = micro_block.get("from_scratch")
                if not fs_block or not isinstance(fs_block, dict):
                    continue

                # fs_block looks like {"from_scratch": [ { "metrics_summary": ... } ]}
                first_list = next(iter(fs_block.values()), [])
                if not first_list:
                    continue
                metrics_summary = first_list[0].get("metrics_summary", {})

                for j_m, mname in enumerate(metrics):
                    stats = metrics_summary.get(mname, {})
                    mean_arr[i, j_m] = stats.get("mean", np.nan)
                    std_arr[i, j_m] = stats.get("std", np.nan)

            out[mode] = {
                "mean": mean_arr,
                "std": std_arr,
                "top_sizes": top_keys,
                "pretrain_sizes": None,  # no pretraining-size axis here
            }

        else:
            # Full 3D tensor: (n_top, n_pretrain_sizes, n_metrics)
            mean_arr = np.full((n_top, n_pre, n_metrics), np.nan, dtype=float)
            std_arr = np.full((n_top, n_pre, n_metrics), np.nan, dtype=float)

            for i, top in enumerate(top_keys):
                micro_block = results[top][model_size]
                mode_block = micro_block.get(mode, {})
                if not isinstance(mode_block, dict):
                    continue

                for j_pre, size in enumerate(pretrain_sizes):
                    size_key = str(size)
                    entry_list = mode_block.get(size_key)
                    if not entry_list:
                        continue

                    metrics_summary = entry_list[0].get("metrics_summary", {})

                    for k_m, mname in enumerate(metrics):
                        stats = metrics_summary.get(mname, {})
                        mean_arr[i, j_pre, k_m] = stats.get("mean", np.nan)
                        std_arr[i, j_pre, k_m] = stats.get("std", np.nan)

            out[mode] = {
                "mean": mean_arr,
                "std": std_arr,
                "top_sizes": top_keys,
                "pretrain_sizes": pretrain_sizes,
            }

    return out

# Top Tagging

## Loading JSON and Plotting

In [ ]:
with open(
    "/global/homes/i/ibrahime/temp/OmniLearnLightining/assets/Results/Top-Class-Finetuned-Model-Paths-Eval.json"
) as f:
    results = json.load(f)

micro_tensors = extract_micro_metric_arrays(results)

clf_mean = micro_tensors["classifier"]["mean"]
clf_std = micro_tensors["classifier"]["std"]

top_sizes = micro_tensors["classifier"]["top_sizes"]
pre_sizes = micro_tensors["classifier"]["pretrain_sizes"]

fs_mean = micro_tensors["from_scratch"]["mean"]
fs_std = micro_tensors["from_scratch"]["std"]

In [ ]:
metric_index = 1
metric_name = "Loss"

modes = [m for m in micro_tensors.keys() if m != "from_scratch"]
if not modes:
    raise ValueError("No non-from_scratch modes found in micro_tensors.")

ref_mode = modes[0]
pre_sizes = micro_tensors[ref_mode]["pretrain_sizes"]
top_sizes = micro_tensors[ref_mode]["top_sizes"]
top_sizes[-1] = "1.2M-Top"
n_top = len(top_sizes)

fs_mean = micro_tensors["from_scratch"]["mean"]
fs_std = micro_tensors["from_scratch"]["std"]

x = np.arange(len(pre_sizes))


def _format_pre_size(s: int) -> str:
    if s == -1:
        return "100M"
    if s >= 1_000_000 and s % 1_000_000 == 0:
        return f"{s // 1_000_000}M"
    if s >= 1_000 and s % 1_000 == 0:
        return f"{s // 1_000}k"
    return str(s)


xlabels = [_format_pre_size(s) for s in pre_sizes]

fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=True)

axes = axes.flatten()

for i, ax in enumerate(axes):
    # Plot each pre-training mode as line+errorbar
    for mode in modes:
        mean_arr = micro_tensors[mode]["mean"]  # shape: [n_top, n_pre, n_metrics]
        std_arr = micro_tensors[mode]["std"]

        y_mean = mean_arr[i, :, metric_index]
        y_std = std_arr[i, :, metric_index]

        ax.errorbar(
            x,
            y_mean,
            yerr=y_std,
            marker="o",
            linestyle="-",
            capsize=10,
            label=mode,
            linewidth=4,
        )

        if i == 0 or i == 2:
            ax.set_ylabel(metric_name)
        if i == 3 or i == 2:
            ax.set_xlabel("Pre-training examples")
    # From-scratch horizontal baseline for this Top size
    baseline = fs_mean[i, metric_index]
    ax.axhline(baseline, linestyle="--", linewidth=4, label="from_scratch")

    ax.set_title(f"Omnilearned-Micro, {top_sizes[i]} Examples")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=0)
    ax.legend()
    if metric_name == "Loss":
        ax.set_yscale("log")
        ax.set_ylim([])
    ax.grid()


plt.show()

In [ ]:
metric_index = -1
metric_name = "Inv Bg Eff"

modes = [m for m in micro_tensors.keys() if m != "from_scratch"]
if not modes:
    raise ValueError("No non-from_scratch modes found in micro_tensors.")

ref_mode = modes[0]
pre_sizes = micro_tensors[ref_mode]["pretrain_sizes"]
top_sizes = micro_tensors[ref_mode]["top_sizes"]
top_sizes[-1] = "1.2M-Top"
n_top = len(top_sizes)

fs_mean = micro_tensors["from_scratch"]["mean"]
fs_std = micro_tensors["from_scratch"]["std"]

# Now x-axis is TOP sizes
x = np.arange(n_top)


def _format_pre_size(s: int) -> str:
    if s == -1:
        return "100M"
    if s >= 1_000_000 and s % 1_000_000 == 0:
        return f"{s // 1_000_000}M"
    if s >= 1_000 and s % 1_000 == 0:
        return f"{s // 1_000}k"
    return str(s)


xlabels = top_sizes  # x-axis labels are now the Top sizes

fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=True)
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(pre_sizes):
        # In case there are fewer pre-training sizes than subplots
        ax.axis("off")
        continue

    pre_label = _format_pre_size(pre_sizes[i])

    # Plot each training mode as line+errorbar across TOP sizes
    for mode in modes:
        mean_arr = micro_tensors[mode]["mean"]  # shape: [n_top, n_pre, n_metrics]
        std_arr = micro_tensors[mode]["std"]

        # Now vary over top index, fix pre-training index = i
        y_mean = mean_arr[:, i, metric_index]
        y_std = std_arr[:, i, metric_index]

        ax.errorbar(
            x,
            y_mean,
            yerr=y_std,
            marker="o",
            linestyle="-",
            capsize=10,
            label=mode,
            linewidth=4,
        )

    # From-scratch curve vs TOP size (same in all subplots)
    fs_y = fs_mean[:, metric_index]
    ax.plot(
        x,
        fs_y,
        linestyle="--",
        linewidth=4,
        label="from_scratch",
    )

    if i == 0 or i == 2:
        ax.set_ylabel(metric_name)
    if i == 2 or i == 3:
        ax.set_xlabel("Top examples")

    ax.set_title(f"Omnilearned-Micro, Pre-train = {pre_label} examples")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=0)
    ax.legend()
    ax.grid()

plt.show()